<center>
    <img src="./images/mlfasp.png">
</center>

#### Prof. Dr. -Ing. Gerald Schuller <br> Jupyter Notebook: Renato Profeta

[Applied Media Systems Group](https://www.tu-ilmenau.de/en/applied-media-systems-group/) <br>
[Technische Universität Ilmenau](https://www.tu-ilmenau.de/)

# Convolutional Autoencoder

In [ ]:
%%html
<iframe width="560" height="315" src="https://www.youtube.com/embed/Oms4nkitLzE?rel=0" frameborder="0" allow="accelerometer; clipboard-write; encrypted-media; gyroscope; picture-in-picture" allowfullscreen></iframe>

A so-called Autoencoder is a neural network with unsupervised training, which means that we don't need to supply a target function. There is only a training set, which is also the target set. A convolutional autoencoder uses convolutional neural networks. 

An autoencoder maps the input signal to a lower dimensional representation using its encoder part. In this way it is similar for instance to an audio encoder, which compresses an audio signal into a representation with fewer bits than the original audio signal:

In [ ]:
%%html
<a href="https://github.com/GuitarsAI/AudioCodingTutorials">https://github.com/GuitarsAI/AudioCodingTutorials</a><br>
<iframe width="560" height="315" src="https://www.youtube.com/embed/videoseries?list=PL6QnpHKwdPYjRWkWLswWmxFrDmj6leRwh" frameborder="0" allow="accelerometer; clipboard-write; encrypted-media; gyroscope; picture-in-picture" allowfullscreen></iframe>

The decoder part of an autoencoder maps the lower dimensional representation back into the higher dimensional representation. This is similar to an audio decoder, which decodes the compressed version back to an audio signal.

The reconstruction after the decoder part should be as close as possible to the original. Hence the original (the training set) is also the target.

In our Python example we simply use one 1D convolutional layer for the encoder. This corresponds to the analysis filter bank of an audio encoder.

In [ ]:
%%html
<a href="https://github.com/GuitarsAI/MRSP_Notebooks">https://github.com/GuitarsAI/MRSP_Notebooks</a><br>
<iframe width="560" height="315" src="https://www.youtube.com/embed/videoseries?list=PL6QnpHKwdPYiDOUHecdZc1WPTnJ-dd0cT" frameborder="0" allow="accelerometer; clipboard-write; encrypted-media; gyroscope; picture-in-picture" allowfullscreen></iframe>

But here we also use an activation function, the "tanh" function, which seems to work better than the sigmoid function in this application, probably because it keeps the signs. We use a "stride" or downsampling factor of N=1024, and filter kernels of size 2N. 

To obtain the same dimension as the input at the encoder output, which is a critical sampled filter bank, we need N filters, or N "out_channels".

But because we want to reduce the dimensionality after encoding, we choose a lower number, which makes it an over-critically sampled filter bank, for instance "out_channels=32". This is similar to an audio coder where we simply drop the filter subbands with the highest frequencies.

We still have to choose the "padding", meaning the number of zeros which we pad before and after our audio signal before the convolution (or rather correlation in Pytorch). To align the output of the decoder with the input signal for the case of symmetric filter kernels (impulse responses), we need a padding of *filter-length/2-1*, or *kernel_size/2-1)*, because then the filtering starts with the first sample at the center of the filter, which corresponds to the the position where the filter outputs the correspondingly filtered sample, for both encoder and decoder.

Hence in Python our encoder convolutional layer example becomes:

```python
self.conv1=nn.Conv1d(in_channels=1, out_channels=32, 
                     kernel_size=2048, stride=1024, padding=1023, 
                     bias=True) #Padding for 'same' filters (kernel_size/2-1)
```

Observe that we have 1 input channel (the audio signal), and 32 output channels (the subbands).

For the decoder part we need a synthesis filter bank, which in neural network literature is also called a transposed convolutional layer, in pytorch "ConvTranspose1d":

```python
self.synconv1=nn.ConvTranspose1d(in_channels=32, out_channels=1, 
                                 kernel_size=2048, stride=1024, padding=1023, bias=True)
```

Observe that for this synthesis filter bank we have 32 input channels (the subbands) and 1 output channel (the reconstructed audio signal).

The function for the encoder is now:

```python
def encoder(self, x):
    #Analysis:
    x = self.conv1(x)
    y = torch.tanh(x)
    return y
```

Observe that here we included the tanh activation function, which can also be seen as a range limiter (between -1 and 1).

For the decoder it is:

```python
def decoder(self, y):
    #Synthesis:
    xrek= self.synconv1(y)
    return xrek
```

For both together it becomes the overall autoencoder:

```python
def forward(self, x):
    y=self.encoder(x)
    #y=torch.round(y/0.125)*0.125
    xrek=self.decoder(y)
    return xrek
```

We put these functions in a class "Convautoenc":

```python
class Convautoenc(nn.Module):
```

For the training we use the input signal also as a target. There we have to limit the length to the signal length produced by the model output, which we can simply obtain by letting the model run once before the training:

```python
model = Convautoenc()
Ypred=model(X)
outputlen=len(Ypred[0,0,:])
Y=X[:,:,:outputlen]
```

Y is the target signal with same length as model output.

As loss function a common choice is the Mean Squared Error, even though it might not be optimal:

```python
loss_fn = nn.MSELoss()
```

The training is then done with with the for loop:

```python
for epoch in range(2000):
    Ypred=model(X)
    loss=loss_fn(Ypred, Y)
    if epoch%10==0:
        print(epoch, loss.item())
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
```

In [ ]:
%%html
<iframe width="560" height="315" src="https://www.youtube.com/embed/0pPrUF08s58?rel=0" frameborder="0" allow="accelerometer; clipboard-write; encrypted-media; gyroscope; picture-in-picture" allowfullscreen></iframe>

In [ ]:
# -*- coding: utf-8 -*-
__author__ = 'Gerald Schuller'
__copyright__ = 'G.S.'

"""
convolutional autoencoder for audio signals.
Gerald Schuller, February 2020.
""";
#Ported to Jupyter Notebooks by Renato Profeta, October 2020

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import numpy as np
import matplotlib.pyplot as plt
import sys
import os
import librosa

In [ ]:
if sys.version_info[0] < 3:
   # for Python 2
   import cPickle as pickle
else:
   # for Python 3
   import pickle
   
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("device=", device) 

In [ ]:
def signal2pytorch(x):
    #Function to convert a signal vector x, like a mono audio signal, into a 3-d Tensor that conv1d of Pytorch expects,
    #https://pytorch.org/docs/stable/nn.html
    #Argument x: a 1-d signal as numpy array
    #input x[batch,sample]
    #output: 3-d Tensor X for conv1d input.
    #for conv1d Input: (N,Cin,Lin), Cin: numer of input channels (e.g. for stereo), Lin: length of signal, N: number of Batches (signals) 
    X = np.expand_dims(x, axis=0)  #add channels dimension (here only 1 channel)
    if len(x.shape)==1: #mono:
        X = np.expand_dims(X, axis=0)  #add batch dimension (here only 1 batch)
    X=torch.from_numpy(X)
    X=X.type(torch.Tensor)
    X=X.permute(1,0,2)  #make batch dimension first
    return X

In [ ]:
class Convautoenc(nn.Module):
    def __init__(self):
        super(Convautoenc, self).__init__()
        #Analysis Filterbank with downsampling of N=1024, filter length of 2N, but only N/2 outputs:
        self.conv1=nn.Conv1d(in_channels=1, out_channels=32, kernel_size=2048, stride=1024, padding=1023, bias=True) #Padding for 'same' filters (kernel_size/2-1)

        #Synthesis filter bank:
        self.synconv1=nn.ConvTranspose1d(in_channels=32, out_channels=1, kernel_size=2048, stride=1024, padding=1023, bias=True)

    def encoder(self, x):
        #Analysis:
        x = self.conv1(x)
        y = torch.tanh(x)
        return y
      
    def decoder(self, y):
        #Synthesis:
        xrek= self.synconv1(y)
        return xrek
      
    def forward(self, x):
        y=self.encoder(x)
        #y=torch.round(y/0.125)*0.125
        xrek=self.decoder(y)
        return xrek

In [ ]:
#alternative: speech:
batch=1
audio, samplerate = librosa.load("./audio/ACDC - Back In Black Intro.wav", mono=False, sr=None, offset=6)
audio[0,:]/=np.abs(audio[0,:]).max()
audio[1,:]/=np.abs(audio[1,:]).max()
X_train=signal2pytorch(audio[0,:]).to(device) #Convert to pytorch format, batch is first dimension    
X_test=signal2pytorch(audio[1,:]).to(device) #Convert to pytorch format, batch is first dimension    

In [ ]:
print("Generate Model:")
model = Convautoenc().to(device)
print('Total number of parameters: %i' % (sum(p.numel() for p in model.parameters() if p.requires_grad)))
print("Def. loss function:")
loss_fn = nn.MSELoss()  #MSE
#loss_fn = nn.L1Loss()
    
Ypred=model(X_train)
   
#Ypred=Ypred.detach()
outputlen=len(Ypred[0,0,:]) #length of the signal at the output of the network.
print("outputlen=", outputlen)
    
Y=X_train[:,:,:outputlen]  #the target signal with same length as model output
    
print("Input X.shape=", X_train.shape )
print("Target Y.shape=", Y.shape)
print("Target Y=", Y)
#print("max(max(Y))=", max(max(max(Y))))
#print("min(min(Y))=", min(min(min(Y))))
print("Y.type()=", Y.type())

In [ ]:
learning_rate = 1e-4
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)#, betas=(0.9, 0.999))
"""
try:
    checkpoint = torch.load("audio_autoenc.torch",map_location='cpu')
    model.load_state_dict(checkpoint['model_state_dict'])
    #optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
except IOError:
    print("fresh start")
""";
    
#optimrandomdir_pytorch.optimizer(model, loss_fn, X, Ypred, iterations=300, startingscale=1.0, endscale=0.0)
Ypred=model(X_train)
#Ypred=Ypred.detach()
print("Ypred=", Ypred)
    
#randdir=True # True for optimization of random direction, False for pytorch optimization
randdir=False
    
if randdir==True:
#optimization of weights using method of random directions:
    optimrandomdir_pytorch.optimizer(model, loss_fn, X_train, Y, iterations=100000, startingscale=0.25, endscale=0.0)
    #--End optimization of random directions------------------------
else:
    for epoch in range(10000):
        Ypred=model(X_train)
        #print("Ypred.shape=", Ypred.shape)
        #loss wants batch in the beginning! (Batch, Classes,...)
        #Ypredp=Ypred.permute(1,2,0)
        #Yp=Y.permute(1,0)
        #print("Ypredp.shape=", Ypredp.shape, "Yp.shape=", Yp.shape )
        loss=loss_fn(Ypred, Y)
        if epoch%10==0:
            print(epoch, loss.item())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

In [ ]:
"""
torch.save({#'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict()}, "audio_autoenc.torch")
"""
    
ww = model.state_dict()   #read obtained weights
print("ww=", ww)
#Plot obtained weights:
plt.figure(figsize=(10,6))
plt.plot(np.transpose(np.array(ww['conv1.weight'][0:1,0,:].cpu())))
plt.plot(np.transpose(np.array(ww['synconv1.weight'][0:1,0,:].cpu())))
plt.legend(('Encoder Analysis filter 0', 'Decoder Filter 0'))
plt.xlabel('Sample')
plt.ylabel('Value')
plt.title('The Encoder and Decoder Filter Coefficients')
plt.grid()

#Test on training set:
predictions=model(X_train).cpu() # Make Predictions based on the obtained weights, on training set
predictions=predictions.detach()
predictions=np.array(predictions)
Y=np.array(Y.cpu()) #target
#print("Y=",Y)
print("predictions.shape=", predictions.shape)
#convert to numpy:
#https://discuss.pytorch.org/t/how-to-transform-variable-into-numpy/104/2
#Plot target signal and output of autoencoder:
plt.figure(figsize=(10,6))
for b in range(batch):
    plt.plot(np.array(Y[b,0,:]))
    plt.plot(predictions[b,0,:])
    plt.legend(('Target','Predicted'))
    plt.title('The Target and Predicted Signal, batch '+str(b))
    plt.xlabel('Sample')
    plt.grid()
xrek=predictions[:,0,:]  #remove unnecessary dimension for playback
#xrek=np.transpose(xrek)
#xrek=np.clip(xrek, -1.0,1.0)

In [ ]:
import IPython.display as ipd
display(ipd.Audio(xrek, rate=samplerate));

Observe that here the difference is clearly big.

Observe that it is very noisy.

In this way, the autoencoder can be seen as a kind of a filter. It learns a low dimensional subspace from the training set. Every new input is mapped onto this subspace. If it doesn't quite fit onto this subspace, major elements might be missing.

In [ ]:
#Test on Verification set:
predictions=model(X_test).cpu() # Make Predictions based on the obtained weights, on verification set
predictions=predictions.detach()
predictions=np.array(predictions)
plt.figure(figsize=(10,6))
for b in range(batch):
    plt.plot(np.array(X_test[b,0,:].cpu()))
    plt.plot(predictions[b,0,:])
    plt.legend(('Original','Predicted'))
    plt.title('The Original and Predicted Signal, batch '+str(b))
    plt.xlabel('Sample')
    plt.grid()
xrek=predictions[:,0,:]

In [ ]:
display(ipd.Audio(xrek, rate=samplerate));

## Effects of Signal Shifts

In [ ]:
%%html
<iframe width="560" height="315" src="https://www.youtube.com/embed/jop168eHqTo?rel=0" frameborder="0" allow="accelerometer; clipboard-write; encrypted-media; gyroscope; picture-in-picture" allowfullscreen></iframe>

Observe the stride we used in our autoencoder. The stride corresponds to the down and up-sampling rate of the corresponding filter bank. This sampling together with the convolution has the effect of processing the signal in blocks of size of the stride. 

If we now shift our input signal by prepending the number of stride zeros at the beginning (hence delaying it by "stride" samples), we will get the same blocks, just one block later. But if we prepend a number of zeros which is not an integer multiple of the stride, the blocks will look different, and hence the result will look different, not just shifted.

In our example we can observe this by testing the trained autoencoder first with **100 zeros** added at the beginning of the training signal. The output appears **noisy**.

In [ ]:
#Test on shifted input:
X_train_shifted_100 = nn.ConstantPad1d(100, 0)(X_train)
predictions=model(X_train_shifted_100).cpu() # Make Predictions based on the obtained weights, on verification set
predictions=predictions.detach()
predictions=np.array(predictions)
xrek=predictions[:,0,:]

In [ ]:
display(ipd.Audio(xrek, rate=samplerate));

Then we add **1024 zeros** at the beginning of the signal. The output is a **bit more clear** again.

In [ ]:
#Test on 1024 samples shifted test set (shift identical to the stride size)
X_train_shifted_1024 = nn.ConstantPad1d(1024, 0)(X_train)
predictions=model(X_train_shifted_1024).cpu() # Make Predictions based on the obtained weights, on verification set
predictions=predictions.detach()
predictions=np.array(predictions)
xrek=predictions[:,0,:]

In [ ]:
display(ipd.Audio(xrek, rate=samplerate));

This behaviour might be desired, if a certain position or timing is important, but often it is not desired. A straightforward remedy is to set stride=1. But this leads to a much higher computational complexity for training and testing or inference. As a compromise, we reduce the stride to a small value, for instance stride=16. This has lower complexity, and not much audible shift sensitivity.

Try it by setting stride=16 in both the encoder analysis part and the decoder synthesis part of the autoencoder network in the program.

Now, also the **other signal** from the verification set is much **less noisy**, as is the 100 samples shifted version, although there is still a little noise audible.

But it takes much longer for the training.